# iGSM-med eval log analysis: correct vs. incorrect

Load an `inspect_ai` eval log produced by `src.eval.run` and split the samples into two
sections: problems the model **solved** and problems it **got wrong** (scored with iGSM's
`true_correct`, i.e. final answer **+ arithmetic + dependencies**).

You need a finished run first, e.g.:

```bash
uv run python -m src.eval.run --model models/gpt2-rope-igsm-pack/final \
    --p-format pq --log-dir logs/eval
```

In [ ]:
from pathlib import Path
from inspect_ai.log import read_eval_log

# Point LOG_DIR at whatever you passed as --log-dir (default: logs/eval).
LOG_DIR = Path('logs/eval')
MODEL_PATH = 'models/gpt2-rope-igsm-pack/final'  # only used to decode ids -> text

logs = sorted(LOG_DIR.glob('*.json'))
assert logs, f'no eval logs in {LOG_DIR} (run src.eval.run --log-dir {LOG_DIR} first)'
LOG_PATH = logs[-1]  # most recent
print('reading', LOG_PATH)
log = read_eval_log(LOG_PATH)
samples = log.samples
print(f'{len(samples)} samples | status={log.status}')

In [ ]:
def verdict(sample):
    '''Return (correct: bool, explanation: str) from the sample's single scorer.'''
    score = next(iter(sample.scores.values()))  # 'igsm_correctness'
    return bool(score.value), (score.explanation or '')

correct = [s for s in samples if verdict(s)[0]]
mistakes = [s for s in samples if not verdict(s)[0]]
print(f'correct={len(correct)}   mistakes={len(mistakes)}   '
      f'accuracy={len(correct) / len(samples):.3f}')

In [ ]:
# Decoder for displaying prompt/generation as text (best-effort; sentinels 222/223/224
# are ordinary GPT-2 token ids, so they show up as small glyph tokens).
from transformers import AutoTokenizer
tok = AutoTokenizer.from_pretrained(MODEL_PATH)

def render(sample, truncate_gen=None):
    md = sample.metadata
    gen_text = sample.output.completion or tok.decode(md['generated_ids'], skip_special_tokens=False)
    if truncate_gen:
        gen_text = gen_text[:truncate_gen]
    return {
        'slice': md['slice'],
        'op': md['op'],
        'gold_answer': md.get('gold_answer'),
        'prompt': tok.decode(md['prompt_ids'], skip_special_tokens=False),
        'generation': gen_text,
    }
print('decoder ready')

## ✅ Cases the model solved correctly

In [ ]:
for i, s in enumerate(correct[:20]):
    r = render(s)
    print(f'[{i}] slice={r["slice"]}  op={r["op"]}  gold_ans={r["gold_answer"]}')
    print(r['generation'])
    print('=' * 70)

## ❌ Cases the model got wrong

`why` is the short explanation the scorer recorded (typically a `true_correct` exception on a
malformed/garbled solution, or `'ok'` when the parse succeeded but a calculation/dependency
mismatched the gold `Problem`).

In [ ]:
for i, s in enumerate(mistakes[:20]):
    ok, why = verdict(s)
    r = render(s, truncate_gen=500)
    print(f'[{i}] slice={r["slice"]}  op={r["op"]}  gold_ans={r["gold_answer"]}  why={why}')
    print('--- generation (truncated) ---')
    print(r['generation'])
    print('=' * 70)

## Per-slice breakdown

In [ ]:
import collections

by = collections.defaultdict(lambda: [0, 0])  # [n_correct, n_total]
for s in samples:
    ok, _ = verdict(s)
    by[s.metadata['slice']][1] += 1
    by[s.metadata['slice']][0] += int(ok)

print(f"{'slice':<24}{'n':>6}{'accuracy':>10}")
for k, (c, t) in sorted(by.items()):
    print(f'{k:<24}{t:>6}{c / t:>10.3f}')

## Digging deeper into a single mistake

For a failure you want to study, unpickle the gold `Problem` (call `ensure_igsm_submodule()`
first) and run `true_correct` yourself to get the full `parser` diagnostics. The base64
`problem` and raw `generated_ids` live in the sample's metadata for exactly this.

In [ ]:
import base64, pickle
from src.data.igsm import ensure_igsm_submodule
ensure_igsm_submodule()
from tools.tools_test import true_correct

idx = 0  # which mistake to inspect
s = mistakes[idx]
problem = pickle.loads(base64.b64decode(s.metadata['problem']))
full = s.metadata['prompt_ids'] + s.metadata['generated_ids']
try:
    ok, my_print, parser = true_correct(full, problem=problem)
    print('correct=', ok)
    # parser carries detailed diagnostics: parser.correct_cal(...), parser.parse(problem), ...
except Exception as e:
    print(f'true_correct raised {type(e).__name__}: {e}')